# Controlled FrameCNN + BiGRU Baseline

Первый полный эксперимент после успешного tiny-overfit.

```text
grayscale mouth clips
-> frame CNN
-> bidirectional GRU
-> 10 word classes
```

Основная проверка выполняется на новом спикере `spk06`, которого нет в train. Лучшая эпоха выбирается по validation macro-F1. Test split не используется.

Запускай все ячейки сверху вниз.

## 1. Подключить Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Найти и распаковать датасет

In [ ]:
import zipfile
from pathlib import Path
import shutil

extract_dir = Path("/content/lipreading_data_framecnn")
expected_dir = extract_dir / "ru_dataset"

if expected_dir.exists():
    print("Dataset already exists:", extract_dir)
else:
    print("Dataset not found, searching zip on Google Drive...")

    zip_names = [
        "colab_lipreading_moderate_v3_landmark.zip",
        "kaggle_lipreading_dataset.zip",
        "colab_lipreading_dataset.zip",
    ]
    matches = []
    for zip_name in zip_names:
        matches.extend(Path("/content/drive").rglob(zip_name))

    print("matches:", matches)

    if len(matches) == 0:
        raise FileNotFoundError("Upload the moderate landmark archive or the original Colab dataset archive to Google Drive first")

    zip_path = matches[0]
    print("zip:", zip_path)

    if extract_dir.exists():
        shutil.rmtree(extract_dir)

    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        for member in zip_ref.infolist():
            fixed_name = member.filename.replace("\\", "/")
            target_path = extract_dir / fixed_name

            if fixed_name.endswith("/"):
                target_path.mkdir(parents=True, exist_ok=True)
                continue

            target_path.parent.mkdir(parents=True, exist_ok=True)
            with zip_ref.open(member) as source, open(target_path, "wb") as target:
                target.write(source.read())

    print("Dataset restored:", extract_dir)

data_root = extract_dir / "ru_dataset"
print("data_root:", data_root)
print("dataset exists:", data_root.exists())

## 3. Настройки, split и vocabulary

In [ ]:
from pathlib import Path
from collections import Counter
import random
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader


def set_reproducible_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


SEED = 42
set_reproducible_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

split_name = 'ml_splits_landmark_moderate_v3'
split_dir = data_root / split_name / 'speaker_top10'
if not (split_dir / 'train.csv').exists():
    split_name = 'ml_splits_mouth_crops_padded_clean_manual_no_spk09_10'
split_dir = data_root / split_name / 'speaker_top10'
if not (split_dir / 'train.csv').exists():
    split_name = 'ml_splits_mouth_crops_padded_no_spk09_10'
    split_dir = data_root / split_name / 'speaker_top10'
if not (split_dir / 'train.csv').exists():
    raise FileNotFoundError(f'Train split was not found under {data_root}')

train_df = pd.read_csv(split_dir / 'train.csv')
val_df = pd.read_csv(split_dir / 'val.csv')
test_df = pd.read_csv(split_dir / 'test.csv')
vocab = (split_dir / 'vocab.txt').read_text(encoding='utf-8').splitlines()
word_to_idx = {word: index for index, word in enumerate(vocab)}
idx_to_word = {index: word for word, index in word_to_idx.items()}

checkpoint_tag = 'moderate_v3' if split_name == 'ml_splits_landmark_moderate_v3' else 'legacy_padded'
checkpoint_dir = Path('/content/drive/MyDrive/lipreading_sem4') / f'controlled_framecnn_bigru_{checkpoint_tag}'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print('device:', device)
print('split_dir:', split_dir)
print('train:', len(train_df), 'speakers:', sorted(train_df['speaker_id'].unique()))
print('validation:', len(val_df), 'speakers:', sorted(val_df['speaker_id'].unique()))
print('test reserved:', len(test_df), 'speakers:', sorted(test_df['speaker_id'].unique()))
print('vocab:', vocab)
print('results directory:', checkpoint_dir)

## 4. Проверить распределение классов

In [ ]:
distribution_rows = []
for word in vocab:
    distribution_rows.append({
        'word': word,
        'train': int((train_df['word'] == word).sum()),
        'validation': int((val_df['word'] == word).sum()),
        'test_reserved': int((test_df['word'] == word).sum()),
    })

distribution_df = pd.DataFrame(distribution_rows)
display(distribution_df)

majority_word = val_df['word'].value_counts().index[0]
majority_baseline_acc = val_df['word'].value_counts().iloc[0] / len(val_df)
print('validation majority baseline:', majority_word, round(majority_baseline_acc, 4))
print('A useful model must beat this baseline without predicting only one word.')

## 5. Загрузка grayscale-видео

In [ ]:
def load_video(path, num_frames=24, size=96, grayscale=False):
    cap = cv2.VideoCapture(str(path))

    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        raise ValueError(f'Cannot read video: {path}')

    frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()

        if not ok:
            continue

        if grayscale:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            frame = cv2.resize(frame, (size, size))
            frame = frame[:, :, None]
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (size, size))

        frame = frame / 255.0
        frames.append(frame)

    cap.release()

    if len(frames) == 0:
        raise ValueError(f'No frames loaded: {path}')

    while len(frames) < num_frames:
        frames.append(frames[-1])

    video = np.stack(frames)
    video = torch.tensor(video, dtype=torch.float32)
    video = video.permute(3, 0, 1, 2)

    return video

## 6. Dataset и DataLoader

In [ ]:
class LipReadingDataset(Dataset):
    def __init__(self, df, data_root, word_to_idx, grayscale=True):
        self.df = df.reset_index(drop=True)
        self.data_root = data_root
        self.word_to_idx = word_to_idx
        self.grayscale = grayscale

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        video_path = self.data_root / row['clip_path']
        video = load_video(video_path, grayscale=self.grayscale)
        label = torch.tensor(self.word_to_idx[row['word']], dtype=torch.long)
        return video, label


train_dataset = LipReadingDataset(train_df, data_root, word_to_idx, grayscale=True)
val_dataset = LipReadingDataset(val_df, data_root, word_to_idx, grayscale=True)

train_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    generator=train_generator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
)

batch_x, batch_y = next(iter(train_loader))
print('train batch video shape:', batch_x.shape)
print('train batch labels shape:', batch_y.shape)
print('pixel min/max/std:', batch_x.min().item(), batch_x.max().item(), batch_x.std().item())
print('labels:', batch_y.tolist())

## 7. Модель FrameCNN + BiGRU

CNN одинаково обрабатывает каждый кадр. BiGRU получает последовательность признаков и учитывает порядок движения губ.

In [ ]:
class FrameCNNBiGRU(nn.Module):
    def __init__(self, num_classes, cnn_dim=64, gru_hidden=64, dropout=0.3):
        super().__init__()
        self.frame_encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, cnn_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(cnn_dim),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )
        self.temporal_encoder = nn.GRU(
            input_size=cnn_dim,
            hidden_size=gru_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(gru_hidden * 2, num_classes),
        )

    def forward(self, video):
        # video: [batch, channels, frames, height, width]
        batch_size, channels, frames, height, width = video.shape
        video = video.permute(0, 2, 1, 3, 4)
        video = video.reshape(batch_size * frames, channels, height, width)
        frame_features = self.frame_encoder(video)
        frame_features = frame_features.reshape(batch_size, frames, -1)
        _, hidden = self.temporal_encoder(frame_features)
        clip_features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(clip_features)


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


set_reproducible_seed(SEED)
model = FrameCNNBiGRU(num_classes=len(vocab)).to(device)
print('trainable parameters:', count_parameters(model))

with torch.no_grad():
    logits = model(batch_x.to(device))
print('logits shape:', logits.shape)

## 8. Метрики и training loop

Checkpoint сохраняется по validation macro-F1. Early stopping срабатывает после семи эпох без улучшения.

In [ ]:
def confusion_matrix_array(preds, labels, num_classes):
    matrix = np.zeros((num_classes, num_classes), dtype=int)
    for true_index, pred_index in zip(labels, preds):
        matrix[true_index, pred_index] += 1
    return matrix


def classification_metrics(preds, labels, num_classes):
    matrix = confusion_matrix_array(preds, labels, num_classes)
    total = int(matrix.sum())
    accuracy = float(np.trace(matrix) / total) if total else 0.0
    class_rows = []

    for class_id in range(num_classes):
        true_positive = int(matrix[class_id, class_id])
        support = int(matrix[class_id, :].sum())
        predicted_count = int(matrix[:, class_id].sum())
        precision = true_positive / predicted_count if predicted_count else 0.0
        recall = true_positive / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        class_rows.append({
            'class_id': class_id,
            'support': support,
            'predicted_count': predicted_count,
            'correct': true_positive,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        })

    active_rows = [row for row in class_rows if row['support'] > 0]
    macro_f1 = float(np.mean([row['f1'] for row in active_rows])) if active_rows else 0.0
    balanced_accuracy = float(np.mean([row['recall'] for row in active_rows])) if active_rows else 0.0
    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'balanced_accuracy': balanced_accuracy,
        'class_rows': class_rows,
        'confusion_matrix': matrix,
    }


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        correct += (outputs.argmax(dim=1) == batch_y).sum().item()
        total += batch_y.size(0)

    return total_loss / total, correct / total


def evaluate_detailed(model, loader, criterion, device, num_classes):
    model.eval()
    total_loss = 0.0
    total = 0
    preds = []
    labels = []

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item() * batch_x.size(0)
            total += batch_y.size(0)
            preds.extend(outputs.argmax(dim=1).cpu().tolist())
            labels.extend(batch_y.cpu().tolist())

    metrics = classification_metrics(preds, labels, num_classes)
    metrics['loss'] = total_loss / total
    metrics['preds'] = preds
    metrics['labels'] = labels
    return metrics


def load_checkpoint(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def fit_controlled_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    num_epochs=30,
    patience=7,
):
    best_checkpoint_path = checkpoint_dir / 'framecnn_bigru_gray_controlled_best.pt'
    history_path = checkpoint_dir / 'framecnn_bigru_gray_controlled_history.csv'
    history_rows = []
    best_macro_f1 = -1.0
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    total_start = time.perf_counter()

    for epoch in range(1, num_epochs + 1):
        epoch_start = time.perf_counter()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = evaluate_detailed(model, val_loader, criterion, device, len(vocab))
        epoch_seconds = time.perf_counter() - epoch_start

        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['accuracy'],
            'val_macro_f1': val_metrics['macro_f1'],
            'val_balanced_acc': val_metrics['balanced_accuracy'],
            'epoch_seconds': epoch_seconds,
        }
        history_rows.append(row)
        pd.DataFrame(history_rows).to_csv(history_path, index=False)

        print(
            f"epoch {epoch}/{num_epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f} "
            f"val_macro_f1={val_metrics['macro_f1']:.4f} "
            f"val_bal_acc={val_metrics['balanced_accuracy']:.4f} | {epoch_seconds:.1f}s"
        )

        improved = (
            val_metrics['macro_f1'] > best_macro_f1
            or (
                val_metrics['macro_f1'] == best_macro_f1
                and val_metrics['loss'] < best_val_loss
            )
        )
        if improved:
            best_macro_f1 = val_metrics['macro_f1']
            best_val_loss = val_metrics['loss']
            epochs_without_improvement = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'vocab': vocab,
                'word_to_idx': word_to_idx,
                'idx_to_word': idx_to_word,
                'best_val_macro_f1': best_macro_f1,
                'best_val_acc': val_metrics['accuracy'],
                'best_val_balanced_accuracy': val_metrics['balanced_accuracy'],
                'best_val_loss': best_val_loss,
                'trainable_parameters': count_parameters(model),
                'history': history_rows,
            }, best_checkpoint_path)
            print('saved best by validation macro-F1:', best_checkpoint_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f'early stop: no macro-F1 improvement for {patience} epochs')
            break

    total_seconds = time.perf_counter() - total_start
    return pd.DataFrame(history_rows), best_checkpoint_path, history_path, total_seconds

## 9. Обучение

Максимум 30 эпох. История и лучший checkpoint сохраняются на Google Drive после каждой эпохи/улучшения.

In [ ]:
set_reproducible_seed(SEED)
model = FrameCNNBiGRU(num_classes=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

history_df, best_checkpoint_path, history_path, total_training_seconds = fit_controlled_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=30,
    patience=7,
)

print('epochs completed:', len(history_df))
print('total training minutes:', round(total_training_seconds / 60, 2))
print('history:', history_path)

## 10. Оценить лучший checkpoint на validation

In [ ]:
checkpoint = load_checkpoint(best_checkpoint_path, device)
best_model = FrameCNNBiGRU(num_classes=len(vocab)).to(device)
best_model.load_state_dict(checkpoint['model_state_dict'])

val_metrics = evaluate_detailed(best_model, val_loader, criterion, device, len(vocab))
print('best epoch:', checkpoint['epoch'])
print('validation accuracy:', round(val_metrics['accuracy'], 4))
print('validation macro-F1:', round(val_metrics['macro_f1'], 4))
print('validation balanced accuracy:', round(val_metrics['balanced_accuracy'], 4))
print('majority baseline accuracy:', round(majority_baseline_acc, 4))

class_report_rows = []
for row in val_metrics['class_rows']:
    class_report_rows.append({
        'word': idx_to_word[row['class_id']],
        'support': row['support'],
        'predicted_count': row['predicted_count'],
        'correct': row['correct'],
        'precision': row['precision'],
        'recall': row['recall'],
        'f1': row['f1'],
    })

class_report_df = pd.DataFrame(class_report_rows)
display(class_report_df)

prediction_counts = Counter(idx_to_word[index] for index in val_metrics['preds'])
print('predicted word distribution:')
print(pd.Series(prediction_counts).sort_values(ascending=False))

validation_predictions_df = val_df.copy()
validation_predictions_df['true_id'] = val_metrics['labels']
validation_predictions_df['pred_id'] = val_metrics['preds']
validation_predictions_df['pred_word'] = [idx_to_word[index] for index in val_metrics['preds']]
validation_predictions_df['correct'] = (
    validation_predictions_df['true_id'] == validation_predictions_df['pred_id']
)

class_report_path = checkpoint_dir / 'framecnn_bigru_gray_controlled_class_report.csv'
predictions_path = checkpoint_dir / 'framecnn_bigru_gray_controlled_val_predictions.csv'
class_report_df.to_csv(class_report_path, index=False)
validation_predictions_df.to_csv(predictions_path, index=False)
print('saved class report:', class_report_path)
print('saved validation predictions:', predictions_path)

## 11. Графики и confusion matrix

In [ ]:
curves_path = checkpoint_dir / 'framecnn_bigru_gray_controlled_curves.png'
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train')
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('epoch')
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['train_acc'], label='train accuracy')
axes[1].plot(history_df['epoch'], history_df['val_acc'], label='validation accuracy')
axes[1].axhline(majority_baseline_acc, color='gray', linestyle='--', label='majority baseline')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('epoch')
axes[1].legend()

axes[2].plot(history_df['epoch'], history_df['val_macro_f1'], label='macro-F1')
axes[2].plot(history_df['epoch'], history_df['val_balanced_acc'], label='balanced accuracy')
axes[2].set_title('Balanced validation metrics')
axes[2].set_xlabel('epoch')
axes[2].legend()

fig.tight_layout()
fig.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print('saved curves:', curves_path)

matrix = val_metrics['confusion_matrix']
confusion_path = checkpoint_dir / 'framecnn_bigru_gray_controlled_confusion.png'
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(matrix, cmap='Blues')
ax.set_title('FrameCNN+BiGRU: speaker validation confusion matrix')
ax.set_xlabel('predicted word')
ax.set_ylabel('true word')
ax.set_xticks(range(len(vocab)), vocab, rotation=45, ha='right')
ax.set_yticks(range(len(vocab)), vocab)
fig.colorbar(image, ax=ax)

for true_index in range(matrix.shape[0]):
    for pred_index in range(matrix.shape[1]):
        if matrix[true_index, pred_index]:
            ax.text(pred_index, true_index, str(matrix[true_index, pred_index]), ha='center', va='center')

fig.tight_layout()
fig.savefig(confusion_path, dpi=150, bbox_inches='tight')
plt.show()
print('saved confusion matrix:', confusion_path)

## 12. Итог эксперимента

In [ ]:
predicted_class_count = len(set(val_metrics['preds']))
controlled_summary = pd.DataFrame([{
    'model': 'FrameCNN+BiGRU grayscale',
    'split': 'speaker-based',
    'train_clips': len(train_df),
    'validation_clips': len(val_df),
    'best_epoch': checkpoint['epoch'],
    'validation_accuracy': val_metrics['accuracy'],
    'validation_macro_f1': val_metrics['macro_f1'],
    'validation_balanced_accuracy': val_metrics['balanced_accuracy'],
    'majority_baseline_accuracy': majority_baseline_acc,
    'predicted_classes': predicted_class_count,
    'trainable_parameters': checkpoint['trainable_parameters'],
    'training_minutes': total_training_seconds / 60,
    'test_used': False,
}])

summary_path = checkpoint_dir / 'framecnn_bigru_gray_controlled_summary.csv'
controlled_summary.to_csv(summary_path, index=False)
display(controlled_summary)
print('saved summary:', summary_path)

if predicted_class_count <= 2:
    print('RESULT: class collapse is still present; inspect macro-F1 and confusion matrix.')
elif val_metrics['macro_f1'] <= 0.10:
    print('RESULT: predictions are diverse, but macro-F1 is near random; model/data improvements are needed.')
else:
    print('RESULT: the model learned more than one class. Compare this baseline with the pretrained ResNet experiment next.')

print('TEST SPLIT WAS NOT USED.')

## Что прислать после запуска

Скачай выполненный notebook и пришли его сюда. Все артефакты также сохраняются в:

```text
/content/drive/MyDrive/lipreading_sem4/controlled_framecnn_bigru
```

Особенно нужны итоговая таблица, prediction distribution, class report и confusion matrix.